In [2]:
import torch

print(torch.__version__)
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")

2.11.0+cu128
True
Tesla T4


In [3]:
!pip install -q ultralytics

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.6/46.6 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 56.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.8/75.8 kB 6.8 MB/s eta 0:00:00


In [4]:
!pip install -q kagglehub

import kagglehub
from pathlib import Path

kitti_path = kagglehub.dataset_download("ibrahimalobaid/kitte-dataset")

kitti_imgs = sorted(
    Path(f"{kitti_path}/training/image_2/").glob("*.png")
)

print("KITTI path:", kitti_path)
print("Number of images:", len(kitti_imgs))

100%|██████████| 5.69G/5.69G [02:22<00:00, 42.9MB/s]

Extracting files...


KITTI path: /root/.cache/kagglehub/datasets/ibrahimalobaid/kitte-dataset/versions/1
Number of images: 7481


In [5]:
from pathlib import Path

image_dir = Path(kitti_path) / "training" / "image_2"
label_dir = Path(kitti_path) / "training" / "label_2"

print("Images:", len(list(image_dir.glob("*.png"))))
print("Labels:", len(list(label_dir.glob("*.txt"))))

print("\nExample label:")
example = next(label_dir.glob("*.txt"))
print(example)
print(example.read_text()[:1000])

Images: 7481
Labels: 7481

Example label:
/root/.cache/kagglehub/datasets/ibrahimalobaid/kitte-dataset/versions/1/training/label_2/006920.txt
Van 0.00 0 1.85 178.21 138.93 492.63 374.00 2.05 2.02 5.18 -2.87 1.78 8.91 1.55
Car 0.71 1 2.19 0.00 210.62 234.86 374.00 1.40 1.59 3.89 -5.27 1.84 6.65 1.54
Car 0.95 0 0.97 884.06 193.28 1241.00 374.00 1.44 1.59 3.51 2.51 1.56 3.03 1.61
Car 0.00 0 -1.69 664.84 177.98 740.54 233.27 1.41 1.64 3.77 2.50 1.58 20.56 -1.57
Car 0.00 0 1.68 510.91 180.25 545.84 204.20 1.43 1.76 3.97 -5.15 1.91 45.72 1.57
Car 0.00 1 1.90 303.42 200.76 434.85 271.45 1.31 1.58 4.23 -5.48 2.03 16.84 1.59
DontCare -1 -1 -10 546.02 171.90 608.56 189.65 -1 -1 -1 -1000 -1000 -1000 -10
DontCare -1 -1 -10 629.35 168.77 654.39 186.52 -1 -1 -1 -1000 -1000 -1000 -10



In [6]:
from ultralytics import YOLO

model = YOLO("/content/best.pt")
print(model.names)

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.
{0: 'Car', 1: 'Truck', 2: 'Bus', 3: 'Motorcycle', 4: 'Bicycle', 5: 'TrafficSigns', 6: 'TrafficLight', 7: 'Pedestrians'}


In [7]:
from pathlib import Path
from PIL import Image
import shutil
import os

# kitti_path already comes from:
# kitti_path = kagglehub.dataset_download("ibrahimalobaid/kitte-dataset")

src_images = Path(kitti_path) / "training" / "image_2"
src_labels = Path(kitti_path) / "training" / "label_2"

out = Path("/content/kitti_yolo_eval")
out_images = out / "images"
out_labels = out / "labels"

out_images.mkdir(parents=True, exist_ok=True)
out_labels.mkdir(parents=True, exist_ok=True)

# KITTI class -> YOUR YOLOv8 class ID
class_map = {
    "Car": 0,
    "Truck": 1,
    "Cyclist": 4,
    "Pedestrian": 7,
}

converted_objects = 0
ignored_objects = 0

images = sorted(src_images.glob("*.png"))

for i, image_path in enumerate(images):
    label_path = src_labels / f"{image_path.stem}.txt"

    with Image.open(image_path) as im:
        width, height = im.size

    yolo_lines = []

    if label_path.exists():
        for line in label_path.read_text().splitlines():
            parts = line.split()

            if len(parts) < 8:
                continue

            cls_name = parts[0]

            if cls_name not in class_map:
                ignored_objects += 1
                continue

            # KITTI 2D bbox:
            # fields 4,5,6,7 = left, top, right, bottom
            x1, y1, x2, y2 = map(float, parts[4:8])

            # Clamp to image boundaries
            x1 = max(0.0, min(x1, width))
            x2 = max(0.0, min(x2, width))
            y1 = max(0.0, min(y1, height))
            y2 = max(0.0, min(y2, height))

            box_w = x2 - x1
            box_h = y2 - y1

            if box_w <= 0 or box_h <= 0:
                continue

            x_center = ((x1 + x2) / 2.0) / width
            y_center = ((y1 + y2) / 2.0) / height
            norm_w = box_w / width
            norm_h = box_h / height

            cls_id = class_map[cls_name]

            yolo_lines.append(
                f"{cls_id} {x_center:.6f} {y_center:.6f} "
                f"{norm_w:.6f} {norm_h:.6f}"
            )

            converted_objects += 1

    # Symlink instead of copying ~7,500 images
    destination = out_images / image_path.name
    if not destination.exists():
        os.symlink(image_path, destination)

    # Empty label file is valid for images with no mapped objects
    (out_labels / f"{image_path.stem}.txt").write_text(
        "\n".join(yolo_lines)
    )

print("Images:", len(images))
print("Converted objects:", converted_objects)
print("Ignored KITTI objects:", ignored_objects)
print("Dataset:", out)

Images: 7481
Converted objects: 35950
Ignored KITTI objects: 15915
Dataset: /content/kitti_yolo_eval


In [10]:
yaml_content = """
path: /content/kitti_yolo_eval

train: images
val: images

names:
  0: Car
  1: Truck
  2: Bus
  3: Motorcycle
  4: Bicycle
  5: TrafficSigns
  6: TrafficLight
  7: Pedestrians
"""

with open("/content/kitti_yolo_eval/dataset.yaml", "w") as f:
    f.write(yaml_content)

print(open("/content/kitti_yolo_eval/dataset.yaml").read())


path: /content/kitti_yolo_eval

train: images
val: images

names:
  0: Car
  1: Truck
  2: Bus
  3: Motorcycle
  4: Bicycle
  5: TrafficSigns
  6: TrafficLight
  7: Pedestrians



In [11]:
from ultralytics import YOLO

model = YOLO("/content/best.pt")

metrics = model.val(
    data="/content/kitti_yolo_eval/dataset.yaml",
    split="val",
    imgsz=640,
    batch=16,
    device=0,
    conf=0.001,
    iou=0.6,
    plots=True,
    project="/content/kitti_evaluation",
    name="native_carla_yolov8s"
)

Ultralytics 8.4.150 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Model summary (fused): 72 layers, 11,128,680 parameters, 0 gradients, 28.4 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 2238.1±1881.9 MB/s, size: 779.4 KB)
val: Scanning /content/kitti_yolo_eval/labels... 7481 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 7481/7481 230.8it/s 32.4s
val: New cache created: /content/kitti_yolo_eval/labels.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 468/468 3.6it/s 2:09
                   all       7481      35950      0.274      0.222      0.167     0.0724
                   Car       6684      28742      0.704      0.507      0.551      0.248
                 Truck       1036       1094      0.104      0.284     0.0537      0.029
               Bicycle       1141       1627     0.0153    0.00369   0.000708   0.000119
           Pedestrians       1779       4487      0.273     0.0944   

In [12]:
print("mAP@50:     ", metrics.box.map50)
print("mAP@50-95:  ", metrics.box.map)
print("mAP@75:     ", metrics.box.map75)
print("Per-class AP:", metrics.box.maps)

mAP@50:      0.16694400997284647
mAP@50-95:   0.0724059781035932
mAP@75:      0.053281555944509264
Per-class AP: [    0.24753    0.029027    0.072406    0.072406  0.00011854    0.072406    0.072406    0.012944]


In [17]:
from ultralytics import YOLO

model_c2r = YOLO("/content/carla2real_best.pt")

metrics_c2r = model_c2r.val(
    data="/content/kitti_yolo_eval/dataset.yaml",
    split="val",
    imgsz=640,
    batch=16,
    device=0,
    conf=0.001,
    iou=0.6,
    plots=True,
    project="/content/kitti_evaluation",
    name="carla2real_yolov8s"
)

print("CARLA2Real mAP@50:   ", metrics_c2r.box.map50)
print("CARLA2Real mAP@50-95:", metrics_c2r.box.map)
print("Per-class AP:", metrics_c2r.box.maps)

Ultralytics 8.4.150 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Model summary (fused): 73 layers, 11,128,680 parameters, 0 gradients, 28.4 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1702.1±1393.1 MB/s, size: 838.7 KB)
val: Scanning /content/kitti_yolo_eval/labels.cache... 7481 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 7481/7481 1.6Git/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 468/468 3.7it/s 2:05
                   all       7481      35950      0.276     0.0593     0.0505     0.0185
                   Car       6684      28742      0.558      0.118      0.143     0.0589
                 Truck       1036       1094     0.0538     0.0622    0.00779    0.00332
               Bicycle       1141       1627     0.0925    0.00492    0.00538    0.00109
           Pedestrians       1779       4487      0.401     0.0524     0.0458     0.0105
Speed: 0.5ms preprocess, 3.2ms inferen